<a href="https://colab.research.google.com/github/sumyuck/ML-learning/blob/main/cvdl/CVDL_p-5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import cv2

DATASET_DIR = Path("DataSet-2classes")

In [ ]:
agri_files = [DATASET_DIR / f"agricultural{i:02d}.tif" for i in range(100)]
air_files  = [DATASET_DIR / f"airplane{i:02d}.tif" for i in range(100)]

paths = agri_files + air_files
labels = [0]*100 + [1]*100

In [ ]:
def gray_histogram(img_path, bins=256):
    img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
    hist = cv2.calcHist([img],[0],None,[bins],[0,256]).flatten()
    hist = hist / (hist.sum() + 1e-8)
    return hist.astype(np.float32)

In [ ]:
X_gray = np.vstack([gray_histogram(p, 256) for p in paths])
y = np.array(labels, dtype=np.int64)
X_gray.shape, y.shape

((200, 256), (200,))

In [ ]:
df_gray = pd.DataFrame(X_gray, columns=[f"gray_{i}" for i in range(X_gray.shape[1])])
df_gray["label"] = y
df_gray.to_csv("features_gray_hist.csv", index=False)
df_gray.head()

,gray_0,gray_1,gray_2,gray_3,gray_4,gray_5,gray_6,gray_7,gray_8,gray_9,...,gray_247,gray_248,gray_249,gray_250,gray_251,gray_252,gray_253,gray_254,gray_255,label
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0


In [ ]:
def hsv_histogram(img_path, bins=(8,8,8)):
    img = cv2.imread(str(img_path))
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    hist = cv2.calcHist([hsv],[0,1,2],None,list(bins),[0,180,0,256,0,256]).flatten()
    hist = hist / (hist.sum() + 1e-8)
    return hist.astype(np.float32)

In [ ]:
X_hsv = np.vstack([hsv_histogram(p, (8,8,8)) for p in paths])
X_hsv.shape

(200, 512)

In [ ]:
df_hsv = pd.DataFrame(X_hsv, columns=[f"hsv_{i}" for i in range(X_hsv.shape[1])])
df_hsv["label"] = y
df_hsv.to_csv("features_hsv_hist.csv", index=False)
df_hsv.head()

,hsv_0,hsv_1,hsv_2,hsv_3,hsv_4,hsv_5,hsv_6,hsv_7,hsv_8,hsv_9,...,hsv_503,hsv_504,hsv_505,hsv_506,hsv_507,hsv_508,hsv_509,hsv_510,hsv_511,label
0,0.0,0.001846,0.038574,0.076187,0.066711,0.005798,0.001526,0.000305,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
1,0.0,0.000092,0.039307,0.118515,0.141144,0.011505,0.000702,0.000183,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
2,0.0,0.000107,0.051819,0.132599,0.114227,0.010513,0.001068,0.000031,0.0,0.000092,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
3,0.0,0.000015,0.038803,0.130936,0.082016,0.004807,0.000168,0.000000,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
4,0.0,0.000458,0.001083,0.004425,0.017563,0.027390,0.000031,0.000000,0.0,0.000015,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0


Now creating a neural network model

In [ ]:
import numpy as np, pandas as pd, torch, torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix

In [ ]:
CSV_PATH = "features_gray_hist.csv"
df = pd.read_csv(CSV_PATH)
X = df.drop(columns=["label"]).values.astype(np.float32)
y = df["label"].values.astype(np.int64)
num_classes = len(np.unique(y))
X.shape, np.bincount(y)

((200, 256), array([100, 100]))

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
scaler = StandardScaler()
X_tr = scaler.fit_transform(X_tr).astype(np.float32)
X_te = scaler.transform(X_te).astype(np.float32)

In [ ]:
Xtr_t = torch.from_numpy(X_tr)
ytr_t = torch.from_numpy(y_tr)
Xte_t = torch.from_numpy(X_te)
yte_t = torch.from_numpy(y_te)

In [ ]:
train_ds = torch.utils.data.TensorDataset(Xtr_t, ytr_t)
test_ds  = torch.utils.data.TensorDataset(Xte_t, yte_t)
train_loader = torch.utils.data.DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader  = torch.utils.data.DataLoader(test_ds, batch_size=256, shuffle=False)

In [ ]:
inp = X.shape[1]
hid = 128
model = nn.Sequential(
    nn.Linear(inp, hid),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(hid, hid//2),
    nn.ReLU(),
    nn.Linear(hid//2, num_classes)
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

Sequential(
  (0): Linear(in_features=256, out_features=128, bias=True)
  (1): ReLU()
  (2): Dropout(p=0.2, inplace=False)
  (3): Linear(in_features=128, out_features=64, bias=True)
  (4): ReLU()
  (5): Linear(in_features=64, out_features=2, bias=True)
)

In [ ]:
opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
crit = nn.CrossEntropyLoss()
epochs = 25
for epoch in range(epochs):
    model.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        loss = crit(model(xb), yb)
        loss.backward()
        opt.step()

In [ ]:
model.eval()
with torch.no_grad():
    preds = []
    trues = []
    for xb, yb in test_loader:
        xb = xb.to(device)
        logits = model(xb)
        preds.append(torch.argmax(logits, dim=1).cpu().numpy())
        trues.append(yb.numpy())
y_pred = np.concatenate(preds)
y_true = np.concatenate(trues)
acc = accuracy_score(y_true, y_pred)
cm = confusion_matrix(y_true, y_pred)
acc, cm

(0.975,
 array([[20,  0],
        [ 1, 19]]))